# FightWatch — đối chứng pipeline v2
Giữ nguyên 109 clip validation của pilot cũ. So sánh interaction, full-frame và full-frame + hard negatives lấy riêng từ train. Cùng seed, optimizer, số epoch tối đa và budget từng nhánh. Chọn checkpoint bằng AP; báo thêm precision–recall theo ngưỡng. Đây là development validation đã dùng nhiều lần, không phải test độc lập. Không tự phê duyệt model demo.
Giữ nguyên 16 frame/8 FPS trong đối chứng này để không thay đổi cửa sổ và nhãn validation. Cấu hình thời gian pretrained sẽ là thí nghiệm riêng. Repo và artifact cũ không bị thay thế.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, shutil, json, zipfile, time, threading
ROOT = Path('/content/drive/MyDrive/SENTINEL')
SOURCE = ROOT / 'artifacts/fight_pilot_x3d_v1'
OUTPUT = ROOT / 'artifacts/fight_ablation_v2'
CACHE = ROOT / 'features/fight_ablation_v2'
DATA = Path('/content/fightwatch-data/UBI_FIGHTS')
REPO = ROOT / 'repo/anomaly-behavior-detector-system'
LOCAL = Path('/content/fightwatch-code')
URL = 'https://github.com/TranDuon/anomaly-behavior-detector-system.git'
BUDGET_MINUTES = 25
for directory in (OUTPUT, CACHE, DATA, REPO.parent):
    directory.mkdir(parents=True, exist_ok=True)
assert (SOURCE / 'manifest.json').exists() and (SOURCE / 'report.json').exists(), 'Thiếu artifact pilot gốc'
if (REPO / '.git').exists():
    remote = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
    assert remote.removesuffix('.git') == URL.removesuffix('.git'), 'Sai remote repo Drive'
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', URL, str(REPO)], check=True)
shutil.copytree(REPO, LOCAL, dirs_exist_ok=True, ignore=shutil.ignore_patterns('.git', '.venv', '__pycache__', 'models', 'uploads', 'logs', '.env', '.pytest_cache'))
os.chdir(LOCAL)
revision = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
(OUTPUT / 'code_revision.txt').write_text(revision)
print('REVISION', revision)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Chọn GPU trước khi chạy tiếp'
print('GPU', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_fight_ablation.py', 'tests/test_fight_runtime.py', '-q', '-p', 'no:cacheprovider'], check=True)


In [ ]:
manifest = json.loads((SOURCE / 'manifest.json').read_text())
selected = sorted({row['key'] for row in manifest['rows']})
archive_path = ROOT / 'datasets/fight/UBI_FIGHTS.zip'
assert archive_path.stat().st_size == 8_117_112_419, 'Archive gốc không khớp kích thước'
with zipfile.ZipFile(archive_path) as archive:
    names = archive.namelist()
    test_name = next(name for name in names if name.endswith('/test_videos.csv') or name == 'test_videos.csv')
    official_test = set(archive.read(test_name).decode().splitlines())
    assert not set(selected) & official_test
    videos = {Path(name).stem: name for name in names if Path(name).suffix.lower() in {'.mp4', '.avi'} and '/videos/' in '/' + name}
    annotations = {Path(name).stem: name for name in names if '/annotation/' in '/' + name and name.endswith('.csv')}
    for index, key in enumerate(selected, 1):
        for member in (videos[key], annotations[key]):
            parts = Path(member).parts
            marker = 'videos' if 'videos' in parts else 'annotation'
            destination = DATA.joinpath(*parts[parts.index(marker):])
            destination.parent.mkdir(parents=True, exist_ok=True)
            if not destination.exists() or destination.stat().st_size != archive.getinfo(member).file_size:
                temporary = destination.with_suffix(destination.suffix + '.partial')
                with archive.open(member) as source, temporary.open('wb') as target:
                    shutil.copyfileobj(source, target)
                temporary.replace(destination)
        if index % 8 == 0:
            print('Prepared', index, '/', len(selected), flush=True)
    (DATA / 'test_videos.csv').write_bytes(archive.read(test_name))
print('Đã chuẩn bị đúng các video của split cũ; không giải nén toàn dataset.')


In [ ]:
command = [sys.executable, '-u', '-m', 'scripts.fight_ablation', '--source-run', str(SOURCE), '--dataset-root', str(DATA), '--output-dir', str(OUTPUT), '--cache-dir', str(CACHE), '--baseline-cache', str(ROOT / 'features/fight_pilot_v1/x3d_s'), '--budget-minutes', str(BUDGET_MINUTES)]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
def stop_over_budget():
    if process.poll() is None:
        process.terminate()
        threading.Timer(10, lambda: process.kill() if process.poll() is None else None).start()
watchdog = threading.Timer(BUDGET_MINUTES * 60 + 120, stop_over_budget)
watchdog.start()
try:
    with (OUTPUT / 'run_log.txt').open('a') as log:
        log.write(f'\nREVISION {revision} | {time.ctime()}\n')
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
    return_code = process.wait()
finally:
    watchdog.cancel()
assert return_code == 0, 'Phiên đối chứng dừng sớm; xem run_log.txt/comparison.json, không tự train tiếp'


In [ ]:
import pandas as pd
from IPython.display import display, Video
comparison = json.loads((OUTPUT / 'comparison.json').read_text())
table = [{'variant': 'baseline_v1', **comparison['baseline']['validation']}]
for name, report in comparison['variants'].items():
    table.append({'variant': name, **report.get('validation', {}), 'decision': report['decision'], 'recall_at_precision75': report.get('best_recall_at_precision75')})
display(pd.DataFrame(table).set_index('variant'))
print('Tất cả model vẫn là PILOT, chưa có test độc lập. Artifact:', OUTPUT)
for name in comparison['variants']:
    error_path = OUTPUT / name / 'errors.json'
    if error_path.exists():
        errors = json.loads(error_path.read_text())
        display(pd.DataFrame(errors).drop(columns=['regions'], errors='ignore'))
        for error in [entry for entry in errors if entry.get('source_clip')][:2]:
            print(name, error['video'], error['start_seconds'], error['kind'], error['score'])
            display(Video(str(OUTPUT / name / error['source_clip']), embed=True, width=420))
            display(Video(str(OUTPUT / name / error['crop_clip']), embed=True, width=420))
